In [ ]:
from hcc_release_paths import project_path as _hcc_path
import pandas as pd
import numpy as np

# ==============================
# Step 2-1：生存信息 QC
# ==============================

clinical_file = _hcc_path('LIHC_Clinical_Survival_Final_371.csv')

clinical = pd.read_csv(clinical_file)

print("=" * 60)
print("TCGA-LIHC Survival QC")
print("=" * 60)

# ------------------------------
# 1. 基本信息
# ------------------------------
print("\n【1. 数据维度】")
print(clinical.shape)

# ------------------------------
# 2. OS_time
# ------------------------------
print("\n【2. OS_time】")

print("缺失数量：", clinical["OS_time"].isna().sum())

print("最小值：", clinical["OS_time"].min())
print("最大值：", clinical["OS_time"].max())
print("中位数：", clinical["OS_time"].median())

# ------------------------------
# 3. OS
# ------------------------------
print("\n【3. OS】")

print(clinical["OS"].value_counts(dropna=False))

# ------------------------------
# 4. OS_time <= 0
# ------------------------------
print("\n【4. OS_time <= 0】")

print(
    (clinical["OS_time"] <= 0).sum()
)

# ------------------------------
# 5. Stage
# ------------------------------
print("\n【5. Stage 分布】")

print(
    clinical["Stage"].value_counts(dropna=False)
)

# ------------------------------
# 6. 关键变量缺失
# ------------------------------
print("\n【6. 关键变量缺失】")

for col in [
    "Patient_ID",
    "Sample_ID",
    "OS_time",
    "OS",
    "Stage",
    "Age",
    "Gender"
]:
    if col in clinical.columns:
        print(
            f"{col:15s}: "
            f"{clinical[col].isna().sum()} missing"
        )

print("\n" + "=" * 60)
print("Survival QC 完成")
print("=" * 60)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter
from tqdm import tqdm
from statsmodels.stats.multitest import multipletests
import os


# ==========================================
# Step 2-2
# TCGA-LIHC 全基因单因素 Cox
# ==========================================


clinical_file = _hcc_path('LIHC_Clinical_Survival_Final_371.csv')

rna_file = _hcc_path('LIHC_RNA_Step1/LIHC_RNA_Normalized_371.tsv')


# -----------------------------
# 1. 读取数据
# -----------------------------

clinical = pd.read_csv(clinical_file)

rna = pd.read_csv(
    rna_file,
    sep="\t",
    index_col=0
)


print("Clinical:", clinical.shape)
print("RNA:", rna.shape)



# -----------------------------
# 2. 生存数据整理
# -----------------------------

surv = clinical[
    [
        "Sample_ID",
        "OS_time",
        "OS"
    ]
].copy()


# 去除缺失
surv = surv.dropna()


print(
    "Survival available:",
    surv.shape[0]
)


# -----------------------------
# 3. RNA 与临床匹配
# -----------------------------

common_samples = list(
    set(surv["Sample_ID"])
    &
    set(rna.columns)
)


print(
    "Matched samples:",
    len(common_samples)
)


# 保留顺序
surv = surv[
    surv["Sample_ID"].isin(common_samples)
]


rna = rna[
    common_samples
]


# 转置
expr = rna.T


expr.index.name = "Sample_ID"


# 合并
data = surv.merge(
    expr,
    on="Sample_ID"
)


print(
    "Final Cox dataset:",
    data.shape
)



# -----------------------------
# 4. 单因素 Cox
# -----------------------------

genes = rna.index.tolist()


results = []


cph = CoxPHFitter()


print("\nStart Cox analysis...")


for gene in tqdm(genes):

    try:

        tmp = data[
            [
                "OS_time",
                "OS",
                gene
            ]
        ].copy()


        tmp.columns = [
            "time",
            "event",
            "expression"
        ]


        # Cox
        cph.fit(
            tmp,
            duration_col="time",
            event_col="event"
        )


        summary = cph.summary.loc[
            "expression"
        ]


        results.append(
            {
                "Gene": gene,
                "HR": summary["exp(coef)"],
                "HR_lower95": summary["exp(coef) lower 95%"],
                "HR_upper95": summary["exp(coef) upper 95%"],
                "P_value": summary["p"]
            }
        )


    except Exception as e:
        pass



cox_result = pd.DataFrame(results)


print(
    "Cox finished:",
    cox_result.shape
)



# -----------------------------
# 5. FDR校正
# -----------------------------

cox_result["FDR"] = multipletests(
    cox_result["P_value"],
    method="fdr_bh"
)[1]


# -----------------------------
# 6. 排序
# -----------------------------

cox_result = cox_result.sort_values(
    "FDR"
)


# -----------------------------
# 7. 保存
# -----------------------------

os.makedirs(
    "LIHC_RNA_Step2",
    exist_ok=True
)


cox_result.to_csv(
    "LIHC_RNA_Step2/Univariate_Cox_AllGenes.csv",
    index=False
)

print("="*60)

print(
    "Significant genes (FDR<0.05):",
    (cox_result["FDR"] < 0.05).sum()
)

print(
    cox_result.head(20)
)

print("="*60)

In [ ]:
import pandas as pd
import os


cox_file = "LIHC_RNA_Step2/Univariate_Cox_AllGenes.csv"


cox = pd.read_csv(cox_file)


# FDR筛选

sig_gene = cox[
    cox["FDR"] < 0.05
].copy()


print(
    "Significant genes:",
    sig_gene.shape
)


# 保存

sig_gene.to_csv(
    "LIHC_RNA_Step2/Significant_Cox_Genes_FDR005.csv",
    index=False
)


sig_gene.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


cox = pd.read_csv(
    "LIHC_RNA_Step2/Univariate_Cox_AllGenes.csv"
)


cox["log2HR"] = np.log2(cox["HR"])

cox["neg_log10_FDR"] = -np.log10(
    cox["FDR"]
)


plt.figure(
    figsize=(8,6)
)


plt.scatter(
    cox["log2HR"],
    cox["neg_log10_FDR"],
    s=10
)


plt.axvline(
    0,
    linestyle="--"
)


plt.axhline(
    -np.log10(0.05),
    linestyle="--"
)


plt.xlabel(
    "log2(Hazard Ratio)"
)


plt.ylabel(
    "-log10(FDR)"
)


plt.title(
    "Univariate Cox Analysis"
)


plt.tight_layout()


plt.savefig(
    "LIHC_RNA_Step2/Volcano_Cox.png",
    dpi=600
)


plt.show()

In [ ]:
import matplotlib.pyplot as plt


cox = pd.read_csv(
    "LIHC_RNA_Step2/Univariate_Cox_AllGenes.csv"
)


top20 = (
    cox
    .sort_values(
        "HR",
        ascending=False
    )
    .head(20)
)



plt.figure(
    figsize=(8,10)
)


plt.errorbar(
    top20["HR"],
    range(len(top20)),
    xerr=[
        top20["HR"]-top20["HR_lower95"],
        top20["HR_upper95"]-top20["HR"]
    ],
    fmt="o"
)


plt.yticks(
    range(len(top20)),
    top20["Gene"]
)


plt.axvline(
    1,
    linestyle="--"
)


plt.xlabel(
    "Hazard Ratio"
)


plt.title(
    "Top Risk Genes"
)


plt.tight_layout()


plt.savefig(
    "LIHC_RNA_Step2/Top20_Risk_Genes.png",
    dpi=600
)


plt.show()